# Predictive Campus Life Wellness Sentinel
## Random Forest - Flask REST API

### Objective
This notebook creates a Flask REST API around the pretrained Random
Forest model.

The API accepts student behavioral information as JSON input, sends it
to the pretrained machine learning model, and returns the predicted
risk level and probability scores as JSON.

### API Endpoint
POST /predict

### Input
Student behavioral features in JSON format.

### Output
Predicted Low / Medium / High risk and probability scores.

In [ ]:
!pip install flask pyngrok

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving wellness_randomforest_pipeline.pkl to wellness_randomforest_pipeline.pkl


In [ ]:
import os

print(os.listdir())

['.config', 'wellness_randomforest_pipeline.pkl', 'sample_data']


In [ ]:
from flask import Flask, request, jsonify
import joblib
import pandas as pd

app = Flask(__name__)

# Load pretrained model
model_package = joblib.load("wellness_randomforest_pipeline.pkl")

model = model_package["model"]
FEATURE_COLUMNS = model_package["features"]
RISK_MAPPING = model_package["risk_mapping"]


@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "message": "Predictive Campus Life Wellness Sentinel API is running"
    })


@app.route("/predict", methods=["POST"])
def predict():

    try:
        # Get JSON input
        data = request.get_json()

        if not data:
            return jsonify({
                "error": "No JSON input provided"
            }), 400

        # Convert input to DataFrame
        input_df = pd.DataFrame([data])

        # Check required features
        missing_features = [
            feature for feature in FEATURE_COLUMNS
            if feature not in input_df.columns
        ]

        if missing_features:
            return jsonify({
                "error": "Missing required features",
                "missing_features": missing_features
            }), 400

        # Keep only model features
        input_df = input_df[FEATURE_COLUMNS]

        # Predict
        predicted_class = model.predict(input_df)[0]

        # Prediction probabilities
        probabilities = model.predict_proba(input_df)[0]

        # Convert class to risk label
        predicted_risk = RISK_MAPPING[predicted_class]

        # JSON response
        response = {
            "predicted_risk": predicted_risk,
            "probabilities": {
                "Low": round(float(probabilities[0]) * 100, 2),
                "Medium": round(float(probabilities[1]) * 100, 2),
                "High": round(float(probabilities[2]) * 100, 2)
            }
        }

        return jsonify(response)

    except Exception as e:

        return jsonify({
            "error": str(e)
        }), 500


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


In [ ]:
from flask import Flask, request, jsonify
import joblib
import pandas as pd

# Create Flask application
app = Flask(__name__)

# Load the pretrained Random Forest model
model_package = joblib.load("wellness_randomforest_pipeline.pkl")

# Extract model information
model = model_package["model"]
FEATURE_COLUMNS = model_package["features"]
RISK_MAPPING = model_package["risk_mapping"]

print("Flask application created successfully!")
print("Model loaded successfully!")
print("Number of features:", len(FEATURE_COLUMNS))

Flask application created successfully!
Model loaded successfully!
Number of features: 14


In [ ]:
@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "message": "Predictive Campus Life Wellness Sentinel API is running"
    })

In [ ]:
@app.route("/predict", methods=["POST"])
def predict():

    try:
        # Receive JSON input
        data = request.get_json()

        # Check whether input was provided
        if not data:
            return jsonify({
                "error": "No JSON input provided"
            }), 400

        # Convert JSON to DataFrame
        input_df = pd.DataFrame([data])

        # Check for missing features
        missing_features = [
            feature for feature in FEATURE_COLUMNS
            if feature not in input_df.columns
        ]

        if missing_features:
            return jsonify({
                "error": "Missing required features",
                "missing_features": missing_features
            }), 400

        # Arrange features in the same order as training
        input_df = input_df[FEATURE_COLUMNS]

        # Make prediction
        predicted_class = model.predict(input_df)[0]

        # Get probabilities
        probabilities = model.predict_proba(input_df)[0]

        # Convert numerical class to risk label
        predicted_risk = RISK_MAPPING[predicted_class]

        # Create JSON response
        return jsonify({
            "predicted_risk": predicted_risk,
            "probabilities": {
                "Low": round(float(probabilities[0]) * 100, 2),
                "Medium": round(float(probabilities[1]) * 100, 2),
                "High": round(float(probabilities[2]) * 100, 2)
            }
        })

    except Exception as e:
        return jsonify({
            "error": str(e)
        }), 500

In [ ]:
import os

print(os.listdir())

['.config', 'wellness_randomforest_pipeline.pkl', 'sample_data']


In [ ]:
import threading

def run_flask():
    app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    )

flask_thread = threading.Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()

print("Flask server started successfully!")

 * Serving Flask app '__main__'
Flask server started successfully!
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000


In [ ]:
import requests

response = requests.get("http://127.0.0.1:5000/")

print("Status Code:", response.status_code)
print("Response:", response.json())

INFO:werkzeug:127.0.0.1 - - [17/Sep/2026 06:13:30] "GET / HTTP/1.1" 200 -


Status Code: 200
Response: {'message': 'Predictive Campus Life Wellness Sentinel API is running'}


In [ ]:
import requests

url = "http://127.0.0.1:5000/predict"

student_data = {
    "facility_usage": 8,
    "dining_activity": 8,
    "event_participation": 4,
    "club_participation": 4,
    "residence_engagement": 9,
    "recreation_activity": 7,
    "social_interactions": 25,
    "communication_activity": 30,
    "sleep_quality": 8,
    "academic_engagement": 9,
    "campus_engagement_score": 85,
    "social_isolation_score": 15,
    "engagement_change_pct": 5,
    "rolling_3_week_engagement": 82
}

response = requests.post(url, json=student_data)

print("Status Code:", response.status_code)
print("Response:")
print(response.json())

INFO:werkzeug:127.0.0.1 - - [17/Sep/2026 06:13:44] "POST /predict HTTP/1.1" 200 -


Status Code: 200
Response:
{'predicted_risk': 'Medium', 'probabilities': {'High': 36.81, 'Low': 1.18, 'Medium': 62.01}}
